# NB6 — Reverse-engineering the QK/OV Circuits (from the weights)

Everything so far has been *behavioural*: we watched activations for specific inputs (patterns,
scores, ablations, patching). This notebook does the **gold-standard** thing in mechanistic
interpretability — read the circuit **directly off the weight matrices**, with no data at all.

Two circuits, matching the two halves of the induction mechanism:

- **OV circuit** — *what a head writes* given what it attends to. We'll show the induction head is a
  **copying head**: its OV circuit maps token `X` back to `X` in logit space.
- **QK circuit** — *where a head attends*. We'll show the prev-token head attends to position `i-1`
  purely from its positional QK circuit.

Then **composition scores** tie them together from the weights alone.

### Why a new (toy) model

Weight-level circuits are only clean in a *clean* model. GPT-2 small has LayerNorm and MLPs that
smear the token↔logit basis — if you compute its OV copying circuit, the copying accuracy is ~1%
(try it!). So for this notebook we switch to a **2-layer attention-only** model (no MLPs, no
LayerNorm, no biases), trained specifically to have clean induction circuits. Same TransformerLens
API, same induction mechanism — just legible at the weights. This is the one place in the course we
leave GPT-2, and for a good reason.

## 0. Load the toy attention-only model

We build it from a `HookedTransformerConfig` and load pretrained weights from HuggingFace (courtesy
of the ARENA course). Notable config choices: `attn_only=True` (no MLPs), `normalization_type=None`
(no LayerNorm), and `positional_embedding_type="shortformer"` (positions are added only into the
Q/K inputs, not the value path — this makes induction heads form more cleanly).

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer, HookedTransformerConfig, FactoredMatrix
from huggingface_hub import hf_hub_download

torch.set_grad_enabled(False)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = HookedTransformerConfig(
    d_model=768, d_head=64, n_heads=12, n_layers=2, n_ctx=2048, d_vocab=50278,
    attention_dir="causal", attn_only=True, tokenizer_name="EleutherAI/gpt-neox-20b",
    seed=398, use_attn_result=True, normalization_type=None,
    positional_embedding_type="shortformer",
)
weights_path = hf_hub_download(repo_id="callummcdougall/attn_only_2L_half", filename="attn_only_2L_half.pth")
model = HookedTransformer(cfg)
model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
model = model.to(device)
print("toy model loaded:", cfg.n_layers, "layers x", cfg.n_heads, "heads")

## 1. Locate the circuit in this model

Different model, different head indices. Let's run the NB3 induction-score check to find *this*
model's induction heads. (It has a known prev-token head at **L0H7**, which we'll use below.)

In [ ]:
seq_len = 50
prefix = torch.zeros(1, 1, dtype=torch.long)                 # a single prefix token
rand = torch.randint(0, model.cfg.d_vocab, (1, seq_len))
rep = torch.cat([prefix, rand, rand], dim=1).to(device)     # [prefix, seq, seq]
_, cache = model.run_with_cache(rep)

print("induction heads (induction-stripe score > 0.4):")
for L in range(model.cfg.n_layers):
    p = cache["pattern", L][0]
    stripe = p.diagonal(offset=-(seq_len - 1), dim1=-2, dim2=-1).mean(-1)
    for H in range(model.cfg.n_heads):
        if stripe[H] > 0.4:
            print(f"  L{L}H{H}: {stripe[H]:.3f}")
print("prev-token head: L0H7 (we'll verify its QK circuit below)")

## 2. The `FactoredMatrix` class

The circuits we want are products like `W_E @ W_V @ W_O @ W_U`, which is `[d_vocab, d_vocab]` =
`50278 x 50278` ≈ 2.5 **billion** entries — too big to materialize. But it's **low rank**: it factors
through `d_head = 64`. TransformerLens's `FactoredMatrix` stores such a product as its two (small)
factors and computes norms, SVDs, and *selected* rows/cols **without ever forming the full matrix**.

A head's OV map `W_V @ W_O` is itself `[d_model, d_model]` but rank `d_head` — a perfect example.

In [ ]:
# W_OV for one head, as a factored (low-rank) matrix — never materialized as [768, 768].
W_OV_1_10 = FactoredMatrix(model.W_V[1, 10], model.W_O[1, 10])   # [d_model, d_head] @ [d_head, d_model]
print("represents shape:", tuple(W_OV_1_10.shape), " (stored as two [768,64] & [64,768] factors)")
print("Frobenius norm (computed without materializing):", round(W_OV_1_10.norm().item(), 2))

## 3. OV circuit — proving the induction head *copies*

The full **OV circuit** of an induction head, in the interpretable token→token basis, is

$$ W_E \; W_{OV} \; W_U \qquad \text{shape } [d_\text{vocab}, d_\text{vocab}] $$

Interpretation: the `(A, B)` entry is *how much having token `A` at the attended position pushes the
logit for predicting `B`*. For a **copying** head this matrix should be **diagonal-dominant** — token
`A` boosts token `A` — because the induction head's job is to copy the attended token to the output.

We measure this with **copying accuracy**: for a sample of tokens, does the OV circuit's row put its
argmax back on the same token? We use `FactoredMatrix` to grab only the sampled rows.

In [ ]:
def ov_copying_accuracy(layer, head, sample=2000, k=1):
    """Fraction of sampled tokens whose OV-circuit row has the same token in its top-k."""
    # full OV circuit [d_vocab, d_vocab], factored through d_head so it's never fully formed:
    full_OV = FactoredMatrix(model.W_E @ model.W_V[layer, head], model.W_O[layer, head] @ model.W_U)
    idx = torch.randint(0, model.cfg.d_vocab, (sample,), device=device)
    rows = full_OV[idx].AB                              # materialize ONLY the sampled rows: [sample, d_vocab]
    topk = rows.topk(k, dim=-1).indices                # [sample, k]
    return (topk == idx[:, None]).any(-1).float().mean().item()

print(f"L1H10 OV copying:  top-1 {ov_copying_accuracy(1, 10):.1%}   top-5 {ov_copying_accuracy(1, 10, k=5):.1%}")
print("(random chance for top-1 is 1/50278 ~ 0.00%, so this is enormously above chance -> it copies.)")

## 4. QK circuit — proving the prev-token head attends one step back

The prev-token head (L0H7) attends to position `i-1`. In this model positions enter only through the
Q/K path (shortformer), so we can read that behaviour off the **positional QK circuit**:

$$ W_\text{pos} \; W_{QK} \; W_\text{pos}^T \qquad W_{QK} = W_Q W_K^T $$

The `(i, j)` entry is the attention score from position `i` (query) to position `j` (key). If we
causal-mask, scale, and softmax it, a prev-token head should light up the **first sub-diagonal**
(`j = i-1`) — a clean stripe of ~1.

In [ ]:
L, H = 0, 7
W_pos = model.W_pos
W_QK = model.W_Q[L, H] @ model.W_K[L, H].T                       # [d_model, d_model]
n = 100
scores = W_pos[:n] @ W_QK @ W_pos[:n].T                          # [n, n] position-by-position
mask = torch.tril(torch.ones_like(scores)).bool()
pattern = torch.where(mask, scores / model.cfg.d_head**0.5, torch.tensor(-1e9, device=device)).softmax(-1)

print(f"L0H7 avg lower-diagonal (attend to previous position): {pattern.diagonal(-1).mean():.4f}")
plt.figure(figsize=(5, 5))
plt.imshow(pattern.cpu().numpy(), cmap="viridis", origin="upper")
plt.xlabel("key position"); plt.ylabel("query position")
plt.title("L0H7 positional QK circuit (prev-token stripe)"); plt.colorbar(); plt.show()

## Practice — is the *other* induction head also a copier?

This model has a second induction head, **L1H4**. Use the `ov_copying_accuracy` helper to check
whether it's also a copying head.

**Your task:**
1. Compute L1H4's OV copying top-1 and top-5 accuracy.
2. Compare to L1H10. You should find L1H4 copies too, but **less strongly** — a real result worth
   noticing: the two induction heads split the work, and aren't identical.

Hint: it's a one-liner per number — `ov_copying_accuracy(1, 4)` and `ov_copying_accuracy(1, 4, k=5)`.

In [ ]:
# TODO(you):
# print top-1 and top-5 OV copying accuracy for L1H4, and compare to L1H10


<details>
<summary>Reference solution (open after you've tried)</summary>

```python
print(f"L1H4  OV copying:  top-1 {ov_copying_accuracy(1, 4):.1%}   top-5 {ov_copying_accuracy(1, 4, k=5):.1%}")
print(f"L1H10 OV copying:  top-1 {ov_copying_accuracy(1, 10):.1%}   top-5 {ov_copying_accuracy(1, 10, k=5):.1%}")
```

Expected: L1H4 around top-1 ~30%, L1H10 around top-1 ~78%. Both far above the ~0.002% chance level,
so both copy — but L1H10 is the stronger copier.

</details>

## 5. Composition scores — the two circuits compose

We proved (from weights) that L0H7 attends to the previous token, and that the L1 heads copy. The
last piece is that they **compose**: the induction head's *keys* read what the prev-token head
*wrote*. The **K-composition score** (same as NB5, now on a clean model) measures exactly this from
the weights:

$$ \text{score} = \frac{\lVert W_{QK}[B] \, W_{OV}[A]^T \rVert_F}{\lVert W_{QK}[B] \rVert_F \, \lVert W_{OV}[A] \rVert_F} $$

Compared to NB5's GPT-2 numbers, the toy model's are cleaner: L0H7 composes with the induction heads
at roughly **3x** its average composition to other layer-1 heads.

In [ ]:
W_OV = model.W_V @ model.W_O                        # [layer, head, d_model, d_model]
W_QK_all = model.W_Q @ model.W_K.transpose(-1, -2)

def k_composition(src, dst):
    A, B = W_OV[src], W_QK_all[dst]
    return ((B @ A.transpose(-1, -2)).norm() / (A.norm() * B.norm())).item()

prev = (0, 7)
for dst in [(1, 4), (1, 10)]:
    print(f"K-comp  L0H7 -> L{dst[0]}H{dst[1]}: {k_composition(prev, dst):.4f}")
baseline = [k_composition(prev, (1, H)) for H in range(model.cfg.n_heads)]
print(f"mean L0H7 -> any layer-1 head: {sum(baseline)/len(baseline):.4f}")

## The circuit, proven from the weights alone

No activations, no inputs — just matrix products:

| Claim | Weight circuit | Result |
|-------|----------------|--------|
| The induction head **copies** the attended token | OV: `W_E W_OV W_U` | diagonal-dominant → ~78% copying accuracy (L1H10) |
| The prev-token head attends **one position back** | QK: `W_pos W_QK W_posᵀ` | clean `i-1` stripe (~1.0) |
| The two **compose** (prev-token feeds induction's keys) | K-composition score | ~3x baseline for L0H7 → induction heads |

This is the deepest form of the induction result: the mechanism is legible in the *static weights*,
independent of any particular input. It complements the behavioural evidence from NB3–NB5 — together
they form a complete, multi-angle proof of the induction circuit.

---
**Done?** Finish the L1H4 practice and you'll have shown *both* induction heads copy, from the
weights.

That completes the course's core arc — you've now analysed the induction circuit five ways
(attention score, DLA, ablation, activation patching, and weight-level circuits). Natural next
explorations: **Q- and V-composition scores** (you should find K dominates for induction), the
**`FactoredMatrix.svd()`** for reading the OV circuit's top singular directions, or applying this
whole toolkit to the **IOI** circuit in GPT-2.